# 🧩 Mini-Lab: When a Class Earns Its Place

**Module: The Python That Runs Your Pipeline** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Explain** what a dataclass is and why it exists — a typed container for related values
2. **Compare** a plain dict, a dataclass, and a full class to decide which fits a given job
3. **Choose** the right shape for a given situation using a simple decision rule

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Dataclass as a Shape | Use a function for behavior and a dataclass for shape: a named, typed bundle of fields that makes pipeline inputs and outputs explicit. Use a class only when you truly need both state and behavior together. |
| Type Hints | Type hints are optional labels on variables and functions that describe what kinds of values you expect, helping tools catch bugs and making code easier to read and refactor. |


## The Problem: Dicts Everywhere

In the previous mini-labs, our `score_documents` function returns `list[dict]`. That works — but what keys does each dict have? What types are the values? You have to read the function body to find out:

```python
results.append({"text": doc, "score": round(overlap, 2)})
```

A dict is **shapeless** — Python doesn't stop you from writing `result["txt"]` (typo) or `result["score"] + result["text"]` (type mismatch). You only find out at runtime.

A **dataclass** gives this bundle of values a name and typed fields so tooling catches mistakes before you run anything.

## Dict vs Dataclass: Side by Side

In [1]:
# --- Dict approach: no structure enforced ---
result_dict = {"text": "Transformers use attention.", "score": 0.71}

# Typo goes unnoticed until runtime
# result_dict["txt"]  # ← KeyError, but nothing warns you beforehand

print("Dict:", result_dict)
print("Access:", result_dict["score"])

Dict: {'text': 'Transformers use attention.', 'score': 0.71}
Access: 0.71


In [2]:
from dataclasses import dataclass


@dataclass
class ScoredDoc:
    """A document with its relevance score."""
    text: str
    score: float


# --- Dataclass approach: named, typed fields ---
result_dc = ScoredDoc(text="Transformers use attention.", score=0.71)

# result_dc.txt  # ← IDE underlines this immediately (AttributeError)

print("Dataclass:", result_dc)
print("Access:   ", result_dc.score)

Dataclass: ScoredDoc(text='Transformers use attention.', score=0.71)
Access:    0.71


Notice what `@dataclass` gives you **for free**:

| Feature | Dict | Dataclass |
|---------|------|-----------|
| Named fields with types | ❌ | ✅ |
| Auto `__init__` | N/A | ✅ |
| Auto `__repr__` | ❌ (prints raw dict) | ✅ (prints class name + fields) |
| IDE auto-complete on fields | ❌ | ✅ |
| Typo caught before runtime | ❌ | ✅ |

## Using a Dataclass in a Function

Let's refactor `score_documents` to return `list[ScoredDoc]` instead of `list[dict]`. The logic is identical — only the container changes.

In [3]:
def score_documents(query: str, documents: list[str], threshold: float = 0.3) -> list[ScoredDoc]:
    """Score documents by keyword overlap. Returns ScoredDoc instances."""
    query_words = set(query.lower().split())
    results = []
    for doc in documents:
        doc_words = set(doc.lower().split())
        overlap = len(query_words & doc_words) / len(query_words)
        if overlap >= threshold:
            results.append(ScoredDoc(text=doc, score=round(overlap, 2)))
    results.sort(key=lambda x: x.score, reverse=True)
    return results


docs = [
    "Transformers use attention to process sequences in parallel.",
    "RNNs handle sequences one token at a time.",
    "Positional encoding tells transformers about word order.",
]

results = score_documents("How do transformers handle long sequences?", docs)
for r in results:
    print(f"  [{r.score}] {r.text}")

The return type `list[ScoredDoc]` tells every caller exactly what fields exist — no guessing, no reading the function body.

## Dataclass Features: Defaults and Frozen

Dataclasses support default values and immutability — two things you'll use often.

In [4]:
@dataclass
class LLMRequest:
    """Configuration for an LLM API call."""
    prompt: str
    model: str = "gpt-4o-mini"
    max_tokens: int = 500
    temperature: float = 0.0


# Only the required field needs to be specified
req1 = LLMRequest(prompt="Explain RAG in one sentence.")
print("Defaults:", req1)

# Override any default
req2 = LLMRequest(prompt="Write a poem.", model="gpt-4o", temperature=0.9)
print("Custom: ", req2)

Defaults: LLMRequest(prompt='Explain RAG in one sentence.', model='gpt-4o-mini', max_tokens=500, temperature=0.0)
Custom:  LLMRequest(prompt='Write a poem.', model='gpt-4o', max_tokens=500, temperature=0.9)


In [5]:
# frozen=True makes instances immutable — good for configs you don't want mutated
@dataclass(frozen=True)
class ChunkConfig:
    """Immutable chunking configuration."""
    chunk_size: int = 500
    overlap: int = 50


cfg = ChunkConfig(chunk_size=300)
print(cfg)

# Trying to mutate raises an error
try:
    cfg.chunk_size = 1000
except AttributeError as e:
    print(f"Mutation blocked: {e}")

ChunkConfig(chunk_size=300, overlap=50)
Mutation blocked: cannot assign to field 'chunk_size'


## When Does a Full Class Earn Its Place?

A dataclass is a **shape** — it holds data. A full class adds **behavior** (methods that do non-trivial work, manage internal state, or control a lifecycle like open/close).

Here's the decision rule:

| Situation | Use |
|-----------|-----|
| A bundle of related values passed around | **Dataclass** |
| A throwaway one-off with 1–2 keys | **Dict** (it's fine) |
| Logic that transforms inputs → outputs | **Pure function** |
| An object that manages state or resources (DB connection, client session) | **Full class** |

Let's see a concrete example where a full class earns its place.

In [6]:
# A full class earns its place when it manages state or resources.
# This token budget tracker has internal state that changes over its lifetime.

class TokenBudget:
    """Tracks token usage against a budget."""

    def __init__(self, limit: int) -> None:
        self.limit = limit
        self._used = 0

    def spend(self, tokens: int) -> None:
        self._used += tokens

    @property
    def remaining(self) -> int:
        return max(0, self.limit - self._used)

    def __repr__(self) -> str:
        return f"TokenBudget(used={self._used}, remaining={self.remaining}, limit={self.limit})"


budget = TokenBudget(limit=1000)
budget.spend(350)
budget.spend(200)
print(budget)
print(f"Remaining: {budget.remaining}")

TokenBudget(used=550, remaining=450, limit=1000)
Remaining: 450


`TokenBudget` has **mutable internal state** (`_used` changes over time) and **behavior** (`spend`, `remaining`). A dataclass wouldn't be the right fit here because the object isn't just holding data — it's *managing* it.

In contrast, `ScoredDoc` and `LLMRequest` are just shapes — they hold values and that's it. Dataclass is the perfect tool for those.

## Quick Decision Checklist

When you're about to create a new container for data, ask these questions in order:

1. **Is it a one-off with 1–2 keys you'll use once?** → Use a **dict**. Don't over-engineer it.
2. **Is it a named bundle of fields passed between functions?** → Use a **dataclass**.
3. **Does it need to manage changing internal state or resources?** → Use a **full class**.
4. **Is it pure logic that transforms inputs to outputs?** → Use a **function** — you don't need a container at all.

## 🎯 Summary

### Key Takeaways

1. **Dataclass = named, typed shape** — `@dataclass` turns a bundle of related values into a proper type with auto-generated `__init__`, `__repr__`, and IDE support
2. **Dict is fine for throwaway data** — but once a dict is passed between functions or appears in a return type, promote it to a dataclass so the shape is explicit
3. **Full class earns its place for state** — use a regular class only when the object manages mutable internal state or resources (connections, budgets, sessions)
4. **Functions stay the default** — if you just need to transform inputs to outputs, a pure function is the right tool; reach for a class or dataclass only when you need to *carry* data